# MapBiomas GIF Factory — piloto
Execute em ordem. O resultado em /content é temporário; monte o Drive para preservá-lo.

In [ ]:
REPO_URL = 'https://github.com/mapbiomas/brazil-fire.git'
REPO_REF = 'main'  # Prefira uma tag ou commit validado
GEE_PROJECT = ''  # Projeto Cloud habilitado no Earth Engine
if not GEE_PROJECT:
    raise ValueError('Preencha GEE_PROJECT antes de executar.')
import os, subprocess, sys
from copy import deepcopy
from google.colab import drive
repo_dir = '/content/brazil-fire'
if not os.path.exists(repo_dir):
    subprocess.run(['git', 'clone', REPO_URL, repo_dir], check=True)
    subprocess.run(['git', '-C', repo_dir, 'checkout', '--detach', REPO_REF], check=True)
else:
    print('Diretório já existe: usando seu conteúdo sem substituir arquivos.')
    print('Confira manualmente se ele corresponde à versão desejada.')
package_dir = os.path.join(repo_dir, 'utils', 'mapbiomas_gif_factory')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', package_dir], check=True)
# Uma instalação editável pode não atualizar o caminho deste kernel em execução.
source_dir = os.path.join(package_dir, 'src')
if source_dir not in sys.path:
    sys.path.insert(0, source_dir)
import ee
from gif_factory.ui import show_ui
from gif_factory.presets import LAYERS as EXAMPLE_LAYERS, TERRITORIES as EXAMPLE_TERRITORIES
ee.Authenticate()
ee.Initialize(project=GEE_PROJECT)
# Comente este bloco se não quiser montar o Drive; /content continua disponível.
try:
    drive.mount('/content/drive')
except Exception as exc:
    print(f'Drive não montado: {exc}. Você ainda pode gerar arquivos em /content.')
print('Preparação concluída.')

In [ ]:
# Camadas próprias: edite estes objetos e execute novamente a interface.
# Os itens pendentes estão visíveis, mas bloqueados até validar bandas, classes e legenda.
LAYERS = [
    {'id': 'lulc_10', 'name': 'Uso e cobertura', 'theme': 'Uso e cobertura', 'collection': 'Coleção 10', 'asset': 'projects/mapbiomas-public/assets/brazil/lulc/collection10/mapbiomas_brazil_collection10_coverage_v2', 'enabled': False, 'reason': 'Confirmar bandas anuais e legenda categórica da coleção 10.'},
    {'id': 'lulc_10_1', 'name': 'Uso e cobertura', 'theme': 'Uso e cobertura', 'collection': 'Coleção 10.1', 'asset': 'projects/mapbiomas-public/assets/brazil/lulc/collection10_1/mapbiomas_brazil_collection10_1_coverage_v1', 'enabled': False, 'reason': 'Confirmar bandas anuais e legenda categórica da coleção 10.1.'},
    {'id': 'fogo4_anual', 'name': 'Área queimada anual', 'theme': 'Fogo', 'collection': 'Coleção 4', 'asset': 'projects/mapbiomas-public/assets/brazil/fire/collection4/mapbiomas_fire_collection4_annual_burned_v1', 'bands': ['burned_area_2022', 'burned_area_2023'], 'visualization': {'min': 0, 'max': 1, 'palette': ['fdfdfd', '800000']}, 'legend': [{'label': 'Não observado', 'color': 'fdfdfd'}, {'label': 'Fogo', 'color': '800000'}], 'status': 'experimental'},
    *[{'id': 'fogo4_' + product, 'name': name, 'theme': 'Fogo', 'collection': 'Coleção 4', 'asset': 'projects/mapbiomas-public/assets/brazil/fire/collection4/mapbiomas_fire_collection4_' + product + '_v1', 'enabled': False, 'reason': 'Confirmar bandas, legenda e eventual transformação específica antes de habilitar.'} for product, name in [
        ('annual_burned_coverage', 'Cobertura queimada anual'),
        ('annual_burned_scar_size_range', 'Tamanho das cicatrizes'),
        ('monthly_burned', 'Área queimada mensal'),
        ('monthly_burned_coverage', 'Cobertura queimada mensal'),
        ('accumulated_burned', 'Área queimada acumulada'),
        ('accumulated_burned_coverage', 'Cobertura queimada acumulada'),
        ('fire_frequency', 'Frequência de fogo'),
        ('fire_frequency_coverage', 'Frequência de fogo por cobertura'),
        ('time_after_fire', 'Tempo após fogo'),
        ('year_last_fire', 'Ano do último fogo'),
    ]],
]
LAYERS

In [ ]:
# Territórios personalizados: disponíveis com qualquer camada selecionada.
# Estes assets precisam estar acessíveis à conta Earth Engine autenticada.
TERRITORIES = [
    {'id': 'matopiba', 'name': 'MATOPIBA', 'asset': 'projects/mapbiomas-workspace/AUXILIAR/ESTATISTICAS/COLECAO8/VERSAO-1/matopiba'},
    {'id': 'matopiba_cerrado', 'name': 'MATOPIBA no Cerrado', 'asset': 'users/dh-conciani/vectors/matopiba_lapig_cerrado'},
]
TERRITORIES

In [ ]:
# Execute novamente esta célula após alterar os arrays.
ui = show_ui(LAYERS, TERRITORIES, project=GEE_PROJECT)